# Travel Planner 完整 E2E 优化教程（textgrad_graph + 端到端评测）

本 Notebook 演示 **完整编排** 上的 prompt 优化：Router/LangGraph → 子 Agent → 聚合回复。

与 `planner_b1_textgrad_graph.ipynb`（L1/L2）的区别：本教程用 **`objective=e2e`**，rollback 与失败筛选都走 `score_e2e_run` 规则分。

## 你将学到什么

1. **E2E 规则分**在整条链路上检查什么（见下表）
2. 规则分（rollback）与 TextGrad loss（改 prompt）如何分工
3. 如何跑 smoke / 正式 E2E 优化，并读懂报告

### E2E 规则分评什么？（`score_e2e_run`）

端到端（E2E）= 用户提问后，系统**真的跑完**拆解 → 路由 → 各子 Agent → 汇总，再按下面规则给 0~1 分。

| 检查项 | 代码/字段名 | 通俗说明 | 举例 |
|--------|-------------|----------|------|
| **最终回复** | `final_response` | 用户最后看到的那段话，不能为空 | 「北京明天晴，适合出行。」 |
| **关键词** | `agent_summary` + `final_response` | 回复或子任务摘要里要出现期望词（如城市、天气） | 含「北京」「天气」 |
| **工具原始结果** | `tool_data` | 子 Agent 调 MCP/API 返回的结构化 JSON，可校验字段 | `{"city": "北京", ...}` |
| **Agent 是否调到** | `subtask_results[*].agent` | 期望的子智能体有没有被执行 | 应出现 `WeatherAgent` |
| **子任务是否完成** | `subtask_results[*].status` | 每个子任务是否标记为完成 | `completed` / `ok` |

> `tool_data` 与 `agent_summary` 的区别：**tool_data** 是工具返回的原始数据；**agent_summary** 是 Agent 用自然语言概括后的文本。规则分对关键词会同时看两者；若 fixture 配置了 `tool_checks`，还会专门查 `tool_data` 里的字段（如 `city` 必须是「北京」）。

权重合计 1.0：有回复 0.10 + 关键词 0.15~0.25 +（可选）tool_data 0.10 + 禁用词 0.15 + Agent 0.35 + 完成数 0.15。详见 `docs/prompt_evolution.md`。

## 四步流程

| Step | 内容 |
|------|------|
| **Step 1** | E2E **基线**评测（优化前） |
| **Step 2** | `textgrad_graph` + `objective=e2e` **优化** |
| **Step 3** | 查看优化报告 |
| **Step 4** | E2E **复评**对比 |

## 前置条件

```powershell
cd Chapter-11
pip install -e ".[evolution]"
pip install nest_asyncio
```

`.env` 配置 API Key；E2E 较慢，建议 `--e2e-timeout 300`。

## 默认 smoke 参数

- `TRAIN_SPLIT=dev`、`DEV_SPLIT=dev`、`MAX_STEPS=1`、`MAX_FAILURE_CASES=1`
- 正式实验：`train` + `MAX_STEPS=3` + `E2E_TIMEOUT=300`

In [1]:
import asyncio
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "agent_framework").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agent_framework.config import create_llm, load_project_dotenv
from agent_framework.optimization.core.save import save_planner_optimization_artifacts
from agent_framework.optimization.decomposition.fixtures import load_decomposition_fixtures
from agent_framework.optimization.e2e.evaluator import evaluate_e2e_benchmark
from agent_framework.optimization.e2e.runtime import build_e2e_orchestrator
from agent_framework.optimization.planner_pipeline import parse_planner_slots, run_planner_optimization
from agent_framework.optimization.prompt_store import load_optimized_prompt_payload, optimized_prompts_path
from domains.travel.prompt_bundle import TravelPrompts
from domains.travel.specs import create_travel_registry_stub

load_project_dotenv()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

OUTPUT_DIR = ROOT / "data" / "benchmark" / "travel_planner"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OPTIMIZED_FILE = optimized_prompts_path("zh")
REPORT_FILE = OUTPUT_DIR / "planner_textgrad_graph_e2e_optimization_report.json"
BASELINE_FILE = OUTPUT_DIR / "baseline_e2e_dev.json"

FIXTURES = load_decomposition_fixtures()
REGISTRY = create_travel_registry_stub()
BASE_PROMPTS = TravelPrompts.build(locale=FIXTURES.locale, use_optimized=False)

EXECUTOR_MODEL = os.getenv("EXECUTOR_MODEL") or os.getenv("DASHSCOPE_CHAT_MODEL") or "qwen-plus"
OPTIMIZER_MODEL = os.getenv("OPTIMIZER_MODEL") or EXECUTOR_MODEL
EXECUTOR_LLM = create_llm(temperature=0, model=EXECUTOR_MODEL)
OPTIMIZER_LLM = create_llm(temperature=0.2, model=OPTIMIZER_MODEL)

TRAIN_SPLIT = "dev"
DEV_SPLIT = "dev"
MAX_STEPS = 1
FAILURE_THRESHOLD = 0.8
MAX_FAILURE_CASES = 1
E2E_PROFILE = "workflow"
E2E_TIMEOUT = 300.0

print(f"项目根: {ROOT}")
print(f"executor={EXECUTOR_MODEL} optimizer={OPTIMIZER_MODEL}")
print(f"E2E timeout={E2E_TIMEOUT}s max_failure_cases={MAX_FAILURE_CASES}")

项目根: D:\myproject\mira-ai-lab\agent-systems-book\Chapter-11
executor=qwen-plus optimizer=qwen-plus
E2E timeout=300.0s max_failure_cases=1


## Step 1　E2E 基线评测（优化前）

跑完整编排，用 `score_e2e_run` 规则分。关注每条 case 的 `details`（含 tool_data 字段检查）。

In [2]:
def build_orchestrator(*, use_optimized: bool = False, overrides: dict | None = None):
    return build_e2e_orchestrator(
        EXECUTOR_LLM,
        locale=FIXTURES.locale,
        profile=E2E_PROFILE,
        enable_memory=False,
        prompt_overrides=overrides,
        use_optimized=use_optimized and not overrides,
    )


async def run_e2e_eval(split: str, *, use_optimized: bool = False):
    overrides = {}
    if use_optimized:
        payload = load_optimized_prompt_payload(FIXTURES.locale)
        overrides = {
            k: str(payload[k])
            for k in ("decomposition_prompt", "agent_routing")
            if str(payload.get(k) or "").strip()
        }
    orchestrator = build_orchestrator(use_optimized=use_optimized, overrides=overrides or None)
    return await evaluate_e2e_benchmark(
        orchestrator,
        fixtures=FIXTURES,
        split=split,
        profile=E2E_PROFILE,
        timeout_sec=E2E_TIMEOUT,
    )


def print_e2e_report(tag: str, report) -> None:
    print(f"\n=== {tag} ===")
    print(f"E2E avg={report.average_score:.3f} cases={report.case_count} profile={report.profile}")
    for item in report.cases:
        status = "PASS" if item.score.total >= FAILURE_THRESHOLD else "FAIL"
        agents = ", ".join(item.invoked_agents) or "(none)"
        print(f"  [{status}] {item.case_id} score={item.score.total:.3f} agents={agents} tool_ok={item.score.tool_data_ok}")
        if item.score.details:
            print(f"    details: {'; '.join(item.score.details)}")
        if item.final_response:
            preview = item.final_response.replace("\n", " ")[:120]
            print(f"    response: {preview}")


step1 = await run_e2e_eval(DEV_SPLIT, use_optimized=False)
print_e2e_report("Step1 E2E 基线", step1)
BASELINE_FILE.write_text(json.dumps(step1.to_dict(), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"\n已保存: {BASELINE_FILE}")

time=2026-06-26T01:06:05.455815+00:00 level=INFO logger=agent_framework.tracing.setup trace_id=- span_id=- msg=tracing.file_exporter output_dir=D:\myproject\mira-ai-lab\agent-systems-book\Chapter-8\traces output_file=D:\myproject\mira-ai-lab\agent-systems-book\Chapter-8\traces\spans_20260626_090605.jsonl
time=2026-06-26T01:06:05.582080+00:00 level=INFO logger=agent_framework.tracing.trace_provider trace_id=287d944df67152ba8d7bfec02eaa9f00 span_id=dcd9f79d7cc80f4b msg=span.start span=latc.multi-agent-platform.request thread_id=case-08 user_query=下周去西安玩 4 天，先查天气再推荐酒店和本地美食
time=2026-06-26T01:06:05.583595+00:00 level=INFO logger=agent_framework.orchestration.router_orchestrator trace_id=287d944df67152ba8d7bfec02eaa9f00 span_id=dcd9f79d7cc80f4b msg=router.request.start domain=travel entry_profile=workflow query_preview=下周去西安玩 4 天，先查天气再推荐酒店和本地美食 thread_id=case-08
time=2026-06-26T01:06:05.585601+00:00 level=INFO logger=agent_framework.tracing.trace_provider trace_id=287d944df67152ba8d7bfec02e

CancelledError: 

## Step 2　E2E textgrad_graph 优化

- `objective="e2e"`：train 上规则分筛失败 → TextGrad 反传 → dev rollback
- `max_failure_cases_per_step`：每步最多 backward 几个最差 case（省 API）

In [ ]:
print(
    f"开始 E2E 优化: train={TRAIN_SPLIT} dev={DEV_SPLIT} max_steps={MAX_STEPS} "
    f"timeout={E2E_TIMEOUT} max_failure_cases={MAX_FAILURE_CASES}"
)
step2 = await run_planner_optimization(
    backend="textgrad_graph",
    slots=parse_planner_slots("all"),
    decomposition_prompt=BASE_PROMPTS.decomposition_prompt,
    agent_routing=BASE_PROMPTS.agent_routing,
    registry=REGISTRY,
    executor_llm=EXECUTOR_LLM,
    optimizer_llm=OPTIMIZER_LLM,
    fixtures=FIXTURES,
    max_steps=MAX_STEPS,
    failure_threshold=FAILURE_THRESHOLD,
    rollback=True,
    train_split=TRAIN_SPLIT,
    dev_split=DEV_SPLIT,
    objective="e2e",
    e2e_profile=E2E_PROFILE,
    e2e_timeout_sec=E2E_TIMEOUT,
    max_failure_cases_per_step=MAX_FAILURE_CASES,
)

save_planner_optimization_artifacts(
    locale=FIXTURES.locale,
    output_path=OPTIMIZED_FILE,
    report_path=REPORT_FILE,
    executor_model=EXECUTOR_MODEL,
    optimizer_model=OPTIMIZER_MODEL,
    backend="textgrad_graph_e2e",
    decomposition_result=step2.decomposition_result,
    routing_result=step2.routing_result,
)

for label, result in (("decomposition", step2.decomposition_result), ("routing", step2.routing_result)):
    if result is None:
        continue
    print(f"{label}: baseline_dev={result.baseline_dev_score:.3f} best_dev={result.best_dev_score:.3f}")

print(f"\n已写入: {OPTIMIZED_FILE}\n报告: {REPORT_FILE}")

## Step 3　查看优化报告

In [ ]:
report = json.loads(REPORT_FILE.read_text(encoding="utf-8")) if REPORT_FILE.is_file() else {}
print(json.dumps(report, ensure_ascii=False, indent=2)[:4000])

## Step 4　E2E 复评（对比 Step 1）

In [ ]:
step4 = await run_e2e_eval(DEV_SPLIT, use_optimized=True)
print_e2e_report("Step4 E2E 优化后", step4)
print(f"\n对比: {step1.average_score:.3f} → {step4.average_score:.3f}")
after_file = OUTPUT_DIR / "after_e2e_dev.json"
after_file.write_text(json.dumps(step4.to_dict(), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"已保存: {after_file}")

## 附录

```powershell
python scripts/eval_travel_e2e.py --split dev
python scripts/optimize_travel_planner.py --backend textgrad_graph --objective e2e --e2e-timeout 300 --max-failure-cases 3
```

详见 `docs/prompt_evolution.md`。